# Theory Comparison: All Five Theories

This notebook runs all 5 group formation theories on the same agent population,
scores them against a ground truth, and explores parameter sensitivity.

**Theories:**
- **CFT** - Consensus-Fracture Theory (threshold cliques)
- **GFT** - Gradient Field Theory (force-based clustering)
- **QST** - Quantum Social Theory (superposition & entanglement)
- **ICT** - Information Cascade Theory (bandwidth-limited communication)
- **TST** - Thermodynamic Social Theory (Potts model)

In [ ]:
import numpy as np
from cft import (
    Agent, Group, TheoryParameters,
    CFT, GFT, QST, ICT, TST,
    PredictionTournament, TheoryComparator,
)
from cft.visualization import (
    plot_groups, plot_theory_comparison, plot_convergence, plot_parameter_sweep,
)

## 1. Setup: Agents and Ground Truth

We create 30 agents in 3 natural clusters (10 agents each) and define the ground truth.

In [ ]:
rng = np.random.default_rng(42)

agents = []
centers = [np.array([0, 0, 0]), np.array([4, 0, 0]), np.array([0, 4, 0])]
for cluster_id, center in enumerate(centers):
    for j in range(10):
        agent_id = cluster_id * 10 + j
        features = center + rng.normal(0, 0.3, 3)
        agents.append(Agent(id=agent_id, features=features))

ground_truth = [
    Group(id=0, members=list(range(0, 10))),
    Group(id=1, members=list(range(10, 20))),
    Group(id=2, members=list(range(20, 30))),
]

params = TheoryParameters(n_agents=30, n_features=3, random_seed=42)
print(f"{len(agents)} agents in {len(ground_truth)} ground-truth groups")

## 2. Prediction Tournament

Run all 5 theories and score their predictions against the ground truth.

In [ ]:
tournament = PredictionTournament(agents, params)
tournament.add_theory("CFT", CFT, threshold=0.5)
tournament.add_theory("GFT", GFT, k=0.3, sigma=2.0)
tournament.add_theory("QST", QST, n_states=5)
tournament.add_theory("ICT", ICT, bandwidth=4)
tournament.add_theory("TST", TST, temperature=0.5, sweeps_per_step=10)

histories = tournament.run(t_max=10.0, dt=1.0)
print("Simulation complete.")

In [ ]:
scores = tournament.score(ground_truth)
for name, s in scores.items():
    print(f"{name}: PAS={s['pas']:.3f}  similarity={s['similarity']:.3f}  "
          f"group_count_accuracy={s['group_count_accuracy']:.3f}  "
          f"size_accuracy={s['size_accuracy']:.3f}")

## 3. Rankings

In [ ]:
ranked = tournament.rankings(ground_truth)
for r in ranked:
    print(f"#{r['rank']} {r['theory']:4s}  PAS={r['score']:.3f}  groups={r['diagnostics']['n_groups']}")

## 4. Visualize Final Groups

In [ ]:
fig = plot_theory_comparison(histories, agents)
fig

## 5. Convergence Analysis

How quickly does each theory converge to stable groups?

In [ ]:
fig = plot_convergence(histories)
fig

## 6. Cross-Theory Agreement (CTAI)

In [ ]:
ctai = tournament.compute_ctai()
print(f"Cross-Theory Agreement Index: {ctai:.3f}")
print("(1.0 = all theories agree perfectly, 0.0 = complete disagreement)")

## 7. Cross-Theory Pairwise Agreement

In [ ]:
analysis = TheoryComparator.analyze_predictions(histories, metric="nmi")
print("Pairwise NMI between theories:")
for pair, nmi in sorted(analysis["cross_theory_agreement"].items()):
    print(f"  {pair}: {nmi:.3f}")

## 8. Parameter Sweep: CFT Threshold

How does the CFT threshold affect group structure?

In [ ]:
fig = plot_parameter_sweep(
    CFT, "threshold", [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
    agents, params, t_max=5.0, dt=1.0,
)
fig

## 9. Parameter Sensitivity Score (PSS)

How stable are predictions under \u00b110% parameter perturbation?

In [ ]:
for theory_name, param_name, base_val in [
    ("CFT", "threshold", 0.5),
    ("GFT", "k", 0.3),
    ("TST", "temperature", 0.5),
]:
    pss = tournament.compute_pss(theory_name, param_name, base_val)
    print(f"{theory_name} ({param_name}={base_val}): PSS = {pss:.3f}")

## 10. Theory Diagnostics

Each theory provides theory-specific diagnostic information.

In [ ]:
result = tournament.results_dict()
for name, diag in result["diagnostics"].items():
    print(f"\n--- {name} ---")
    for k, v in diag.items():
        if isinstance(v, float):
            print(f"  {k}: {v:.4f}")
        else:
            print(f"  {k}: {v}")